# Task 6 – Feature Selection and Recommendation Quality Evaluation

## Objective

To experiment with different combinations of features and identify which combination produces more relevant TED Talk recommendations.

## What We Will Do

In this task, we will compare four feature combinations:

1. Title only
2. Title + Tags
3. Title + Description + Tags
4. Title + Description + Transcript + Tags

We will generate recommendations, compare their results, evaluate their quality, and select the best feature combination.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the cleaned TED Talks dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Display the first five rows
df.head()

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript,combined_text
0,1,averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis ['alternative ener...
1,92,the best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['africa', 'asia', 'google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...","the best stats you've ever seen ['africa', 'as..."
2,7,simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...","simplicity sells ['computers', 'entertainment'..."
3,53,greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['macarthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"in an emotionally charged talk, macarthur-winn...",if you're here today — and i'm very happy that...,"greening the ghetto ['macarthur grant', 'activ..."
4,66,do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,"do schools kill creativity? ['children', 'crea..."


In [2]:
df.columns.tolist()

['talk_id',
 'title',
 'speaker_1',
 'all_speakers',
 'occupations',
 'about_speakers',
 'views',
 'recorded_date',
 'published_date',
 'event',
 'native_lang',
 'available_lang',
 'comments',
 'duration',
 'topics',
 'related_talks',
 'url',
 'description',
 'transcript',
 'combined_text']

"We identified the required text columns in the cleaned TED Talks dataset for conducting four feature selection experiments."

## Experiment 1 – Title Only



In [3]:
titles = df["title"].fillna("")

title_vectorizer = TfidfVectorizer(stop_words="english")

title_tfidf = title_vectorizer.fit_transform(titles)

title_tfidf.shape

(4005, 4847)

For each experiment, we need similarity scores to generate recommendations and compare their relevance.
So, cosine similarity is the next step after creating the TF-IDF matrix. We will calculate it separately for each experiment.

In [4]:
title_cosine_sim = cosine_similarity(title_tfidf)

title_cosine_sim.shape

(4005, 4005)

### Generating Recommendations
In this step, we select five TED Talks as input talks and generate the top 5 similar TED Talk recommendations using only titles.
The recommendations are based on cosine similarity scores.


In [7]:
def recommend_title_only(title, top_n=5):
    if title not in df["title"].values:
        print("TED Talk title not found.")
        return pd.DataFrame()

    talk_index = df[df["title"] == title].index[0]

    similarity_scores = list(enumerate(title_cosine_sim[talk_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = [
        score for score in similarity_scores
        if score[0] != talk_index
    ]

    top_talks = similarity_scores[:top_n]

    results = df.iloc[[i[0] for i in top_talks]][
        ["title", "topics"]
    ].copy()

    results["similarity_score"] = [i[1] for i in top_talks]

    return results.reset_index(drop=True)

In [8]:
selected_talks = df["title"].iloc[:5].tolist()

for talk in selected_talks:
    print("\nSelected TED Talk:", talk)
    display(recommend_title_only(talk, top_n=5))


Selected TED Talk: averting the climate crisis


,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.488037
1,the global learning crisis -- and what to do a...,"['africa', 'children', 'education', 'future', ...",0.293363
2,the state of the climate — and what we might d...,"['business', 'climate change', 'economics', 'g...",0.285390
3,why you should be a climate activist,"['sustainability', 'climate change', 'environm...",0.285390
4,why i must speak out about climate change,"['big problems', 'climate change', 'environmen...",0.261663



Selected TED Talk: the best stats you've ever seen


,title,topics,similarity_score
0,the best kindergarten you’ve ever seen,"['tedx', 'architecture', 'children', 'design',...",0.692006
1,the sea we've hardly seen,"['tedx', 'biodiversity', 'biology', 'microbiol...",0.462250
2,cooking as never seen before,"['creativity', 'design', 'entertainment', 'foo...",0.371158
3,turning powerful stats into art,"['art', 'beauty', 'business', 'culture', 'phot...",0.309727
4,surprising stats about child carseats,"['business', 'cars', 'culture', 'economics', '...",0.295392



Selected TED Talk: simplicity sells


,title,topics,similarity_score
0,toward a science of simplicity,"['chemistry', 'complexity', 'design', 'history...",0.561431
1,designing for simplicity,"['art', 'design', 'simplicity', 'technology']",0.491009
2,averting the climate crisis,"['alternative energy', 'cars', 'climate change...",0.000000
3,the best stats you've ever seen,"['africa', 'asia', 'google', 'demo', 'economic...",0.000000
4,greening the ghetto,"['macarthur grant', 'activism', 'business', 'c...",0.000000



Selected TED Talk: greening the ghetto


,title,topics,similarity_score
0,averting the climate crisis,"['alternative energy', 'cars', 'climate change...",0.0
1,the best stats you've ever seen,"['africa', 'asia', 'google', 'demo', 'economic...",0.0
2,simplicity sells,"['computers', 'entertainment', 'interface desi...",0.0
3,do schools kill creativity?,"['children', 'creativity', 'culture', 'dance',...",0.0
4,behind the design of seattle's library,"['architecture', 'collaboration', 'culture', '...",0.0



Selected TED Talk: do schools kill creativity?


,title,topics,similarity_score
0,4 lessons in creativity,"['art', 'creativity', 'entertainment', 'photog...",0.424686
1,our failing schools. enough is enough!,"['business', 'children', 'education']",0.394853
2,fashion and creativity,"['business', 'creativity', 'design', 'entertai...",0.393244
3,don't kill your language,"['tedx', 'culture', 'language', 'poetry']",0.384004
4,is space trying to kill us?,"['ted-ed', 'space', 'animation', 'universe', '...",0.350704


## Experiment 2 – Title + Tags

### Objective
To generate TED Talk recommendations using both title and tags.

### What We Are Doing
In this experiment, we combine the title and topics (tags) of each TED Talk. We convert this combined text into TF-IDF vectors and calculate cosine similarity,

In [9]:
title_tags_text = (
    df["title"].fillna("") + " " +
    df["topics"].fillna("")
)

title_tags_vectorizer = TfidfVectorizer(stop_words="english")

title_tags_tfidf = title_tags_vectorizer.fit_transform(title_tags_text)

title_tags_tfidf.shape

(4005, 4932)

In [10]:
title_tags_cosine_sim = cosine_similarity(title_tags_tfidf)

title_tags_cosine_sim.shape

(4005, 4005)

In [11]:
def recommend_with_similarity(title, similarity_matrix, top_n=5):
    if title not in df["title"].values:
        print("TED Talk title not found.")
        return pd.DataFrame()

    talk_index = df[df["title"] == title].index[0]

    similarity_scores = list(enumerate(similarity_matrix[talk_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = [
        score for score in similarity_scores
        if score[0] != talk_index
    ]

    top_talks = similarity_scores[:top_n]

    results = df.iloc[[i[0] for i in top_talks]][
        ["title", "topics"]
    ].copy()

    results["similarity_score"] = [i[1] for i in top_talks]

    return results.reset_index(drop=True)

In [12]:
for talk in selected_talks:
    print("\nSelected TED Talk:", talk)
    display(
        recommend_with_similarity(
            talk,
            title_tags_cosine_sim,
            top_n=5
        )
    )


Selected TED Talk: averting the climate crisis


,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.494476
1,the case for optimism on climate change,"['climate change', 'weather', 'science', 'alte...",0.445807
2,the future of cars,"['alternative energy', 'cars', 'design', 'ener...",0.437836
3,fusion is energy's future,"['alternative energy', 'climate change', 'ener...",0.435302
4,why i must speak out about climate change,"['big problems', 'climate change', 'environmen...",0.433480



Selected TED Talk: the best stats you've ever seen


,title,topics,similarity_score
0,new insights on poverty,"['africa', 'asia', 'google', 'economics', 'glo...",0.517878
1,let my dataset change your mindset,"['medicine', 'presentation', 'science', 'techn...",0.413789
2,asia's rise -- how and when,"['asia', 'economics', 'health', 'india', 'stat...",0.386620
3,the global goals we've made progress on -- and...,"['global issues', 'policy', 'statistics', 'eco...",0.335088
4,the best kindergarten you’ve ever seen,"['tedx', 'architecture', 'children', 'design',...",0.326516



Selected TED Talk: simplicity sells


,title,topics,similarity_score
0,designing for simplicity,"['art', 'design', 'simplicity', 'technology']",0.585485
1,toward a science of simplicity,"['chemistry', 'complexity', 'design', 'history...",0.570368
2,tidying up art,"['art', 'design', 'entertainment', 'humor', 'p...",0.395604
3,software (as) art,"['art', 'entertainment', 'invention', 'music',...",0.356083
4,"less stuff, more happiness","['consumerism', 'culture', 'design', 'happines...",0.348514



Selected TED Talk: greening the ghetto


,title,topics,similarity_score
0,winning the oil endgame,"['macarthur grant', 'business', 'economics', '...",0.372561
1,"rebuilding a neighborhood with beauty, dignity...","['macarthur grant', 'activism', 'children', 'c...",0.281729
2,an illustrated journey through rome,"['macarthur grant', 'ancient world', 'art', 'c...",0.273213
3,juggling as art ... and science,"['macarthur grant', 'dance', 'entertainment', ...",0.256823
4,buildings that blend nature and city,"['architecture', 'community', 'nature', 'citie...",0.253960



Selected TED Talk: do schools kill creativity?


,title,topics,similarity_score
0,our failing schools. enough is enough!,"['business', 'children', 'education']",0.390893
1,creative ways to get kids to thrive in school,"['children', 'education', 'parenting', 'teachi...",0.340156
2,a father-daughter dance ... in prison,"['tedx', 'activism', 'children', 'dance', 'par...",0.324032
3,kids need structure,"['tedx', 'children', 'culture', 'education', '...",0.320407
4,use data to build better schools,"['data', 'education', 'global issues', 'inequa...",0.310389


## Experiment 3 – Title + Description + Tags

### Objective
To generate TED Talk recommendations using titles, descriptions, and tags.
In this experiment, we combine the title, description, and topics (tags) of each TED Talk. We convert this combined text into TF-IDF vectors and calculate cosine similarity

In [13]:
title_desc_tags_text = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["topics"].fillna("")
)

title_desc_tags_vectorizer = TfidfVectorizer(stop_words="english")

title_desc_tags_tfidf = title_desc_tags_vectorizer.fit_transform(
    title_desc_tags_text
)

title_desc_tags_tfidf.shape

(4005, 21509)

In [14]:
title_desc_tags_cosine_sim = cosine_similarity(title_desc_tags_tfidf)

title_desc_tags_cosine_sim.shape

(4005, 4005)

In [15]:
for talk in selected_talks:
    print("\nSelected TED Talk:", talk)
    display(
        recommend_with_similarity(
            talk,
            title_desc_tags_cosine_sim,
            top_n=5
        )
    )


Selected TED Talk: averting the climate crisis


,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.345719
1,what comes after an inconvenient truth?,"['alternative energy', 'climate change', 'ener...",0.315296
2,the case for optimism on climate change,"['climate change', 'weather', 'science', 'alte...",0.314076
3,the state of the climate — and what we might d...,"['business', 'climate change', 'economics', 'g...",0.297983
4,why i must speak out about climate change,"['big problems', 'climate change', 'environmen...",0.279556



Selected TED Talk: the best stats you've ever seen


,title,topics,similarity_score
0,let my dataset change your mindset,"['medicine', 'presentation', 'science', 'techn...",0.338809
1,new insights on poverty,"['africa', 'asia', 'google', 'economics', 'glo...",0.290051
2,the good news of the decade? we're winning the...,"['africa', 'tedx', 'data', 'health', 'inequali...",0.247076
3,asia's rise -- how and when,"['asia', 'economics', 'health', 'india', 'stat...",0.224815
4,how statistics can be misleading,"['animation', 'ted-ed', 'education', 'statisti...",0.214255



Selected TED Talk: simplicity sells


,title,topics,similarity_score
0,toward a science of simplicity,"['chemistry', 'complexity', 'design', 'history...",0.306043
1,designing for simplicity,"['art', 'design', 'simplicity', 'technology']",0.237113
2,the music wars,"['entertainment', 'humor', 'music', 'technology']",0.202588
3,the radical promise of the multi-touch interface,"['demo', 'design', 'interface design', 'techno...",0.196044
4,the best computer interface? maybe ... your hands,"['senses', 'ted fellows', 'design', 'interface...",0.180057



Selected TED Talk: greening the ghetto


,title,topics,similarity_score
0,winning the oil endgame,"['macarthur grant', 'business', 'economics', '...",0.201810
1,3 stories of local eco-entrepreneurship,"['tedx', 'business', 'cities', 'community', 'e...",0.194447
2,everyday inventions,"['macarthur grant', 'collaboration', 'design',...",0.171712
3,a teacher growing green in the south bronx,"['tedx', 'business', 'education', 'garden', 'g...",0.153200
4,"rebuilding a neighborhood with beauty, dignity...","['macarthur grant', 'activism', 'children', 'c...",0.151738



Selected TED Talk: do schools kill creativity?


,title,topics,similarity_score
0,bring on the learning revolution!,"['children', 'creativity', 'education', 'inven...",0.350813
1,how to escape education's death valley,"['culture', 'education', 'teaching', 'student']",0.258536
2,the child-driven education,"['children', 'education', 'technology', 'web',...",0.146402
3,let's raise kids to be entrepreneurs,"['tedx', 'business', 'children', 'education', ...",0.142206
4,how teachers can help kids find their politica...,"['education', 'teaching', 'children', 'tedx', ...",0.141751


## Experiment 4 – Title + Description + Transcript + Tags

### Objective
To generate TED Talk recommendations using titles, descriptions, transcripts, and tags.
In this experiment, we combine the title, description, transcript, and topics (tags) of each TED Talk. We convert this combined text into TF-IDF vectors and calculate cosine similarity.

In [16]:
title_desc_transcript_tags_text = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["transcript"].fillna("") + " " +
    df["topics"].fillna("")
)

title_desc_transcript_tags_vectorizer = TfidfVectorizer(
    stop_words="english"
)

title_desc_transcript_tags_tfidf = (
    title_desc_transcript_tags_vectorizer.fit_transform(
        title_desc_transcript_tags_text
    )
)

title_desc_transcript_tags_tfidf.shape

(4005, 71160)

In [17]:
title_desc_transcript_tags_cosine_sim = cosine_similarity(
    title_desc_transcript_tags_tfidf
)

title_desc_transcript_tags_cosine_sim.shape

(4005, 4005)

In [18]:
for talk in selected_talks:
    print("\nSelected TED Talk:", talk)
    display(
        recommend_with_similarity(
            talk,
            title_desc_transcript_tags_cosine_sim,
            top_n=5
        )
    )


Selected TED Talk: averting the climate crisis


,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.263033
1,how we could change the planet's climate future,"['climate change', 'sustainability', 'environm...",0.239241
2,a climate solution where all sides can win,"['big problems', 'business', 'china', 'economi...",0.237136
3,the disarming case to act right now on climate...,"['climate change', 'environment', 'sustainabil...",0.231320
4,climate change is happening. here's how we adapt,"['climate change', 'energy', 'green', 'policy'...",0.230428



Selected TED Talk: the best stats you've ever seen


,title,topics,similarity_score
0,let my dataset change your mindset,"['medicine', 'presentation', 'science', 'techn...",0.568498
1,new insights on poverty,"['africa', 'asia', 'google', 'economics', 'glo...",0.406044
2,the good news of the decade? we're winning the...,"['africa', 'tedx', 'data', 'health', 'inequali...",0.359717
3,"global population growth, box by box","['culture', 'data', 'demo', 'global issues', '...",0.339655
4,religions and babies,"['tedx', 'data', 'global issues', 'population'...",0.334469



Selected TED Talk: simplicity sells


,title,topics,similarity_score
0,nerdcore comedy,"['collaboration', 'comedy', 'community', 'cult...",0.270696
1,cool tricks your phone can do,"['culture', 'technology', 'best of the web']",0.265957
2,design and discovery,"['design', 'exploration', 'humor', 'photograph...",0.263305
3,6 ways to save the internet,"['internet', 'tedx', 'business', 'future', 'in...",0.239696
4,designing for simplicity,"['art', 'design', 'simplicity', 'technology']",0.231638



Selected TED Talk: greening the ghetto


,title,topics,similarity_score
0,"the next big thing is coming from the bronx, a...","['food', 'creativity', 'collaboration', 'cultu...",0.365398
1,a teacher growing green in the south bronx,"['tedx', 'business', 'education', 'garden', 'g...",0.325777
2,how public spaces make cities work,"['architecture', 'cities']",0.262515
3,3 stories of local eco-entrepreneurship,"['tedx', 'business', 'cities', 'community', 'e...",0.250637
4,how students of color confront impostor syndrome,"['children', 'family', 'education', 'community...",0.224110



Selected TED Talk: do schools kill creativity?


,title,topics,similarity_score
0,how to escape education's death valley,"['culture', 'education', 'teaching', 'student']",0.361343
1,bring on the learning revolution!,"['children', 'creativity', 'education', 'inven...",0.348651
2,nerdcore comedy,"['collaboration', 'comedy', 'community', 'cult...",0.275301
3,a one-man world summit,"['comedy', 'global issues', 'government', 'per...",0.274697
4,our failing schools. enough is enough!,"['business', 'children', 'education']",0.267786


## Final Comparison and Recommendation Quality Evaluation

### Comparison of Four Experiments

| Experiment | Features Used | Observation |
|---|---|---|
| Experiment 1 | Title only | Some recommended talks are related, but some are not closely related. |
| Experiment 2 | Title + Tags | Recommendations are more related to the selected talk's topics. |
| Experiment 3 | Title + Description + Tags | Recommendations are more meaningful because they use the title, topics, and description. |
| Experiment 4 | Title + Description + Transcript + Tags | More information is used, but some recommendations are not closely related. |

### Observations

1. Using only titles gives limited information.
2. Adding tags helps find talks about similar topics.
3. Adding descriptions gives better information about each talk.
4. Adding transcripts does not always improve the recommendations.

### Final Conclusion

After comparing the results of all four experiments using five selected TED Talks, I prefer **Experiment 3 – Title + Description + Tags**.

It gives useful recommendations by considering the title, topics, and description. Based on my manual review, this combination gives a good balance of relevant information.

